# OCR Lab — komparatív vizualizáció

**Sprint:** `image_rag_OCR` · **Dátum:** 2026-06-05

Ez a notebook a [`research_design.md`](research_design.md) → [`score_template.md`](score_template.md) → [`decision.md`](decision.md) láncon végigment komparatív kutatás eredményeit vizualizálja egy **3D koordinátarendszerben**.

## Tengelyek és pontméret

- **X = minőség** (3 különböző operacionalizációban — lásd alább)
- **Y = sebesség** (sec/oldal, **log-skálán**, mert PyMuPDF4LLM 0.35s vs MinerU 29s = 80×)
- **Z = mennyiség** (kibányászott kontextus mértéke)
- **Pont méret = pipeline-segítő hatás** = `cross_step_Σ / 12 + capability_count / 5` (két komponens 0–1-re normalizálva, additív összegzés — 0–2 tartomány)
- **Szín = backend**

## Granularitás — két plot párhuzamosan

Minden 3D-ábránál **két nézet**:
- **Bal (4 pont):** backend-szintű overview — agregált értékek (átlag/max)
- **Jobb (60 pont):** backend × oldal részletes — szórás és kilógók láthatóak

## A 3 tengely-verzió

| Verzió | X (minőség) | Y (sebesség) | Z (mennyiség) | Jellege |
|---|---|---|---|---|
| **A** | Manual rubric score (0–3) | sec/page log | char ratio Claude Read-hez (1.0 = baseline) | Egyszerű, szubjektív szakértői ítélet a tengelyen |
| **B** | Composite quality (additív, normalizált) | sec/page log | Extracted units (chars/1000 + n_images + n_formulas + n_headings) | Több komponensből aggregált — **additív** modellt használ |
| **C** | Char ratio (objektív, mért) | sec/page log | Cross-step value (0–12) | Manual score nélkül, csak számolt adatokon |

## ⚠️ Bizonytalanság előzetes jelzések

- **Manual scores** (Verzió A, B): szakértői ítélet 15 oldalas mintán, n=4 backend × 7+ dimenzió. Bizonytalanság ±0.5 pont.
- **Claude Read time** = NEM mért, csak becslés (~30s/oldal kézi gépeléssel). A diagrammon csak Verzió A,B,C label-eljük, de NEM hasonlítható közvetlenül batch-időkhöz.
- **MinerU time** = `total / pages_in_range` átlag (forrás-szintű batch), NEM tényleges per-oldal — egy oldal önmagában is ~5–60s model-init után. Konzervatív becslés.
- **Per-page minőség** (60-pontos plot): a manual score broadcast-olva backend-szintről minden 15 oldalra → minden oldalra ugyanaz a manual quality érték; a per-page variancia a `char ratio` tengelyen látszik csak. A 60-pontos plot tehát részben félrevezető, ha csak X-et nézzük.
- **n=15 oldal** kis sample — a hipotézis-irányok tájékozóak, NEM statisztikailag bizonyítottak (még).

---
## 1. Adat-betöltés

Forrás:
- [`metrics.json`](metrics.json) — runner kimenete (idő, char-count per page per backend)
- [`../../../test_outputs/_ocr_lab/atg_1_het/input_manifest.json`](../../../test_outputs/_ocr_lab/atg_1_het/input_manifest.json) — page → forrás-típus mapping
- [`score_template.md`](score_template.md) — manual scores (hardcode-olva alább)

In [ ]:
# Conda env: play_env
# Ha a notebookot kernel nélkül futtatod, telepítendők: pip install plotly pandas

import json
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Robusztus ROOT-keresés: a notebook .claude/sprints/image_rag/ocr_lab/-ban él,
# de futtathatjuk a repo gyökeréből is (CWD-szerinti detektálás)
CWD = Path.cwd()
candidates = [CWD] + list(CWD.parents)
ROOT = next((c for c in candidates if (c / "test_outputs/_ocr_lab/atg_1_het").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Nem található a test_outputs/_ocr_lab/atg_1_het — futtasd a repo gyökeréből")
print(f"ROOT: {ROOT}")
LAB = ROOT / "test_outputs/_ocr_lab/atg_1_het"
METRICS = LAB / "metrics.json"
MANIFEST = LAB / "input_manifest.json"
MINERU_DIR = LAB / "mineru"

metrics = json.loads(METRICS.read_text(encoding="utf-8"))
pages = json.loads(MANIFEST.read_text(encoding="utf-8"))["pages"]
print(f"Backends in metrics: {[k for k in metrics if k != '_meta']}")
print(f"Pages in manifest: {len(pages)}")

In [ ]:
# Manual scores a score_template.md alapján (frissítve a MinerU magyar mérés után: hu=3)
MANUAL_SCORES = {
    "tesseract":   {"quality_en": 2, "quality_hu": 1, "cross_step": 3,  "capability_count": 1, "install_cost": 2},
    "pymupdf4llm": {"quality_en": 3, "quality_hu": 3, "cross_step": 7,  "capability_count": 3, "install_cost": 0,
                    "quality_scanned": 0},  # csak born-digitalra jó
    "mineru":      {"quality_en": 3, "quality_hu": 3, "cross_step": 12, "capability_count": 5, "install_cost": 3},
    "claude_read": {"quality_en": 3, "quality_hu": 3, "cross_step": 11, "capability_count": 5, "install_cost": 0},
}

# Extracted units backend-szinten (MinerU strukturált egységei a _content_list.json-ból)
# Más backendre approximation: PyMuPDF4LLM = csak chars; Tesseract = csak chars; Claude Read = chars + (megjegyzett) caption-számolás
EXTRACTED_UNITS_PER_SOURCE = {
    # Backend-specifikus aggregates a 4 forrás-szintű output-ból
    # (Verzió B Z tengelyhez használjuk, source-szinten broadcast-olva)
    "mineru": {
        "chattopadhyay2013_paper": {"chars": 5025,  "n_images": None, "n_formulas": None, "n_headings": None},
        "gravdahl1999_chapter":    {"chars": 82564, "n_images": None, "n_formulas": None, "n_headings": None},
        "nagyi2013_eloadas":       {"chars": 12446, "n_images": None, "n_formulas": None, "n_headings": None},
        "tavakoli2004_paper":      {"chars": 7852,  "n_images": None, "n_formulas": None, "n_headings": None},
    }
}

# Pontosabb n_images / n_formulas / n_headings számolás a _content_list.json-ból
for src_stem in EXTRACTED_UNITS_PER_SOURCE["mineru"]:
    p = MINERU_DIR / src_stem / "auto" / f"{src_stem}_content_list.json"
    if p.exists():
        items = json.loads(p.read_text(encoding="utf-8"))
        n_img  = sum(1 for it in items if it.get("type") == "image")
        n_form = sum(1 for it in items if it.get("type") in ("interline_equation", "inline_equation"))
        n_head = sum(1 for it in items if it.get("text_level") is not None)
        EXTRACTED_UNITS_PER_SOURCE["mineru"][src_stem].update(
            n_images=n_img, n_formulas=n_form, n_headings=n_head)

pd.DataFrame(EXTRACTED_UNITS_PER_SOURCE["mineru"]).T

In [ ]:
# DataFrame építés: backend × page granularitás (60 sor)
rows = []
page_lookup = {p["id"]: p for p in pages}

for backend, data in metrics.items():
    if backend == "_meta" or not data.get("available"):
        continue
    times = data.get("times_per_page", {})
    chars = data.get("char_counts", {})
    for page_id, t in times.items():
        p = page_lookup.get(page_id, {})
        rows.append({
            "backend": backend,
            "page_id": page_id,
            "source":  p.get("source", ""),
            "page":    p.get("page"),
            "type":    p.get("type", ""),
            "lang":    p.get("lang", ""),
            "tag":     p.get("tag", ""),
            "sec_per_page": t,
            "chars":        chars.get(page_id, None),
        })

df = pd.DataFrame(rows)

# Char ratio Claude Read-hez (objektív minőség proxy)
baseline = df[df["backend"] == "claude_read"].set_index("page_id")["chars"]
df["char_ratio_to_claude_read"] = df.apply(
    lambda r: r["chars"] / baseline[r["page_id"]] if pd.notna(r["chars"]) and baseline.get(r["page_id"], 0) else None,
    axis=1,
)

# Backend-szintű manual score-okat broadcast-oljuk page-szintre, nyelv alapján
def manual_quality(row):
    s = MANUAL_SCORES[row["backend"]]
    if row["backend"] == "pymupdf4llm" and row["type"] == "scanned":
        return s.get("quality_scanned", 0)
    if row["lang"].startswith("hu"):
        return s["quality_hu"]
    return s["quality_en"]

df["manual_quality"] = df.apply(manual_quality, axis=1)
df["cross_step"]       = df["backend"].map(lambda b: MANUAL_SCORES[b]["cross_step"])
df["capability_count"] = df["backend"].map(lambda b: MANUAL_SCORES[b]["capability_count"])
df["install_cost"]     = df["backend"].map(lambda b: MANUAL_SCORES[b]["install_cost"])

# Pipeline-impact index (pont méret): cross_step/12 + capability/5 (additív, 0-2 tartomány)
df["impact_index"] = df["cross_step"] / 12 + df["capability_count"] / 5

df.head()

### Additív vs. multiplikatív kompozit — miért az additív?

Az `impact_index = cross_step/12 + capability/5` egy **additív modell**. Ez azt jelenti:

- Ha az egyik komponens **0** (pl. egy backend egyetlen capability-vel sem rendelkezik), az index NEM lesz 0 — a másik komponens még hozzájárul.
- A komponensek **kompenzálhatják** egymást: gyenge cross-step value, erős capability set → közepes index.

**Multiplikatív alternatíva** lenne: `impact_index = (cross_step/12) * (capability/5)`. Ez:

- Ha bármelyik komponens 0 → az index 0 ("vétójog").
- Erősebben jutalmazza azokat a backendeket, amelyek **mindkét dimenzión jól szerepelnek**, és bünteti az egyoldalúakat.

**Miért additív itt?** A cross-step és capability dimenziók **különböző tulajdonságokat** mérnek (downstream-érték vs. funkcionális lefedettség). Egy backend lehet hasznos akkor is, ha csak az egyik dimenzión erős — pl. a PyMuPDF4LLM cross-step=7, capability=3, de a born-digital esetekre kiváló. Multiplikatív szorzásnál ez a részhasznosság elveszne.

Ha valaki szigorúbb "all-rounder"-szűrőt szeretne, a Verzió B alatti `composite_quality` magában már multiplikatív tendenciájú (geometriai átlag rokonát közelíti normalizálva), és bemutatjuk azt is.

In [ ]:
# Backend-szintű agregátum (4 sor) — overview plot-okhoz
agg = df.groupby("backend").agg(
    sec_per_page=("sec_per_page", "mean"),
    chars_mean=("chars", "mean"),
    char_ratio_mean=("char_ratio_to_claude_read", "mean"),
    manual_quality_mean=("manual_quality", "mean"),
    cross_step=("cross_step", "first"),
    capability_count=("capability_count", "first"),
    install_cost=("install_cost", "first"),
    impact_index=("impact_index", "first"),
).reset_index()
agg

---
## 2. Verzió A — Manual rubric × log-time × char ratio

**Tengelyek:**
- X: manual quality score (0–3, szakértői rubrika)
- Y: log10(sec/page)
- Z: char ratio Claude Read-hez (1.0 = baseline)

**Olvasási útmutató:** az **ideális backend** a jobb-fent-fent sarokban van (magas minőség, *alacsony* log-idő, magas char-ratio) — **nagy ponttal** (high pipeline-impact).

In [ ]:
import numpy as np

COLOR = {"tesseract":"#1f77b4", "pymupdf4llm":"#2ca02c", "mineru":"#d62728", "claude_read":"#9467bd"}
MARK  = {"scanned":"diamond", "born_digital":"circle"}

def size_scale(idx, lo=8, hi=40):
    """impact_index (0..2) → marker size (8..40)"""
    return lo + (idx / 2.0) * (hi - lo)

def hover_overview(r):
    return (f"<b>{r['backend']}</b><br>"
            f"manual quality (mean): {r['manual_quality_mean']:.2f}<br>"
            f"sec/page (mean): {r['sec_per_page']:.2f}<br>"
            f"char ratio: {r['char_ratio_mean']:.2f}<br>"
            f"cross_step Σ: {r['cross_step']}/12<br>"
            f"capability: {r['capability_count']}/5<br>"
            f"impact_index: {r['impact_index']:.2f}<br>"
            f"install_cost: {r['install_cost']}/3<br>"
            f"⚠️ n=15 oldal sample")

def hover_detail(r):
    return (f"<b>{r['backend']}</b> · {r['page_id']}<br>"
            f"forrás: {r['source']} (p{r['page']}, {r['lang']}, {r['type']})<br>"
            f"manual quality: {r['manual_quality']}<br>"
            f"sec/page: {r['sec_per_page']:.2f}<br>"
            f"chars: {int(r['chars']) if pd.notna(r['chars']) else '—'}<br>"
            f"char ratio: {r['char_ratio_to_claude_read']:.2f}" if pd.notna(r['char_ratio_to_claude_read']) else f"char ratio: —")

def make_3d_pair(df_overview, df_detail, x_col, y_col, z_col,
                 x_title, y_title, z_title, title_main, y_log=True):
    fig = make_subplots(rows=1, cols=2,
                        specs=[[{"type":"scene"}, {"type":"scene"}]],
                        subplot_titles=("Overview: 4 backend (agregált)",
                                        "Detail: 4 backend × 15 oldal (60 pont)"))
    # Overview
    for backend, sub in df_overview.groupby("backend"):
        y_vals = np.log10(sub[y_col]) if y_log else sub[y_col]
        fig.add_trace(go.Scatter3d(
            x=sub[x_col], y=y_vals, z=sub[z_col],
            mode="markers+text",
            text=[backend], textposition="top center",
            marker=dict(size=size_scale(sub["impact_index"].iloc[0]),
                        color=COLOR[backend], opacity=0.85,
                        line=dict(width=1, color="#333")),
            name=backend, legendgroup=backend,
            hovertext=sub.apply(hover_overview, axis=1),
            hoverinfo="text",
        ), row=1, col=1)
    # Detail
    for backend, sub in df_detail.groupby("backend"):
        y_vals = np.log10(sub[y_col]) if y_log else sub[y_col]
        fig.add_trace(go.Scatter3d(
            x=sub[x_col], y=y_vals, z=sub[z_col],
            mode="markers",
            marker=dict(size=size_scale(sub["impact_index"].iloc[0]) * 0.6,
                        color=COLOR[backend], opacity=0.55,
                        symbol=[MARK.get(t, "circle") for t in sub["type"]],
                        line=dict(width=0.5, color="#333")),
            name=backend, legendgroup=backend, showlegend=False,
            hovertext=sub.apply(hover_detail, axis=1),
            hoverinfo="text",
        ), row=1, col=2)

    y_lab = f"log10({y_title})" if y_log else y_title
    fig.update_layout(
        title=title_main,
        scene=dict(xaxis_title=x_title, yaxis_title=y_lab, zaxis_title=z_title),
        scene2=dict(xaxis_title=x_title, yaxis_title=y_lab, zaxis_title=z_title),
        legend=dict(orientation="h", y=-0.05),
        height=600, width=1300, margin=dict(l=0, r=0, t=70, b=0),
    )
    return fig

fig_a = make_3d_pair(
    agg, df,
    x_col="manual_quality_mean", y_col="sec_per_page", z_col="char_ratio_mean",
    x_title="Manual quality (0-3)",
    y_title="sec/page",
    z_title="char ratio (vs Claude Read)",
    title_main="Verzió A — Manual rubric × log-time × char ratio",
)
# A detail plot Z-jét per-page char ratio-val frissítjük (felülírjuk a fenti broadcast-ot)
fig_a.data[4].z = df[df["backend"]=="tesseract"]["char_ratio_to_claude_read"]
fig_a.data[5].z = df[df["backend"]=="pymupdf4llm"]["char_ratio_to_claude_read"]
fig_a.data[6].z = df[df["backend"]=="mineru"]["char_ratio_to_claude_read"]
fig_a.data[7].z = df[df["backend"]=="claude_read"]["char_ratio_to_claude_read"]
fig_a.show()

**Verzió A értelmezése:**

- A **Claude Read** és **MinerU** mind a 3 tengelyen kiváló (quality=3, char ratio=1.0).
- A **Claude Read** gyors a Y-tengelyen (mert a mért érték a runner render-ideje), de ez **félrevezető** — a tényleges szöveg-kinyerés kézi, nem mért → ⚠️ hover-tip.
- A **MinerU** ~30s/oldal log10≈1.5, ami a *legnagyobb* a 4 között → Pareto: minőség↑, gyorsaság↓.
- A **PyMuPDF4LLM** scanned oldalakon (`type=scanned`) a per-page char ratio-ja **rendkívül alacsony** (gravdahl: 57 char / 1700 char → 0.03) — a detail-plotban a bal alsó sarokba esnek. Ezeknek a marker-szimbóluma `diamond` (lásd `MARK` mapping).
- **Tesseract** nagyi (magyar) oldalakon char_ratio ≈ 0.5 → middle-back a Z tengelyen, miközben az X-en (quality=1) is balra esik. Ez a 0.59 átlagos ratio (lásd `score_template.md`).

---
## 3. Verzió B — Composite quality × log-time × Extracted units

**Tengelyek:**
- X: composite quality = `(manual_q / 3 + char_ratio + capability_count / 5) / 3` (additív, normalizált, 0–1)
- Y: log10(sec/page)
- Z: extracted units = `chars/1000 + n_images + n_formulas + n_headings` (csak MinerU-ra ismert; más backend: chars/1000 + 0)

**Megjegyzés az additív kompozícióról:** az X tengelyen 3 komponens egyenletesen súlyozott összegét vesszük. Multiplikatív verzióban (`manual_q/3 * char_ratio * capability/5`) egyetlen 0-érték nullára szorozza az egészet — itt nem ezt akarjuk, mert minden backendnek van legalább egy *erős* oldala.

In [ ]:
# Composite quality (additív, normalizált)
df["composite_quality"] = (df["manual_quality"]/3.0
                          + df["char_ratio_to_claude_read"].fillna(0)
                          + df["capability_count"]/5.0) / 3.0

# Extracted units — csak MinerU-ra van strukturált; más backendre chars-only
def extracted_units(row):
    base = (row["chars"] or 0) / 1000
    if row["backend"] == "mineru":
        # MinerU forrás-szintű strukturált egységek elosztva oldalanként
        src_stem = row["source"].replace(".pdf", "")
        u = EXTRACTED_UNITS_PER_SOURCE["mineru"].get(src_stem, {})
        n_pages_in_src = df[(df["backend"]=="mineru") & (df["source"]==row["source"])].shape[0]
        if n_pages_in_src:
            base += ((u.get("n_images") or 0)
                   + (u.get("n_formulas") or 0)
                   + (u.get("n_headings") or 0)) / n_pages_in_src
    return base

df["extracted_units"] = df.apply(extracted_units, axis=1)

agg_b = df.groupby("backend").agg(
    sec_per_page=("sec_per_page", "mean"),
    composite_quality_mean=("composite_quality", "mean"),
    extracted_units_mean=("extracted_units", "mean"),
    cross_step=("cross_step", "first"),
    capability_count=("capability_count", "first"),
    install_cost=("install_cost", "first"),
    impact_index=("impact_index", "first"),
).reset_index()
agg_b

In [ ]:
fig_b = make_3d_pair(
    agg_b, df,
    x_col="composite_quality_mean", y_col="sec_per_page", z_col="extracted_units_mean",
    x_title="Composite quality (0-1, additív)",
    y_title="sec/page",
    z_title="Extracted units / page (chars/1000 + structures)",
    title_main="Verzió B — Composite quality × log-time × extracted units",
)
# Detail plot per-page composite_quality + extracted_units
for i, backend in enumerate(["tesseract","pymupdf4llm","mineru","claude_read"]):
    sub = df[df["backend"]==backend]
    fig_b.data[4+i].x = sub["composite_quality"]
    fig_b.data[4+i].z = sub["extracted_units"]
fig_b.show()

**Verzió B értelmezése:**

- A **MinerU** Z-tengelyen átlagosan a legmagasabb (chars + strukturált egységek: képek, formulák, headingek). Ez az „érdemi plusz downstream-nek" mérhető formája (H2 hipotézis).
- A **Claude Read** Z-tengelyen csak chars/1000 — nincs strukturált extract. Ezért az extracted_units tengelyen alatta marad a MinerU-nak, **noha** a minősége megegyezik.
- A **PyMuPDF4LLM** scanned-en (detail plot) majdnem nullára esik Z-ben (gravdahl ~57 char/oldal = 0.057 units).
- A composite quality X-tengely nem lineáris a manual-score X-szel (Verzió A), mert a char_ratio és capability komponensek erősebben szétválasztják a backendek-et.

---
## 4. Verzió C — Char ratio × log-time × Cross-step value

**Tengelyek (manual score nélkül — csak számolt adatok):**
- X: char ratio Claude Read-hez (objektív minőség proxy)
- Y: log10(sec/page)
- Z: cross-step value (0–12, score_template-ből)

Ez a verzió **a legkevésbé szubjektív** — a manual score-t teljesen kikerüli az X tengelyen. A Z tengelyen a cross-step érték persze szakértői ítélet, de **egyetlen** dimenziós.

In [ ]:
fig_c = make_3d_pair(
    agg, df,
    x_col="char_ratio_mean", y_col="sec_per_page", z_col="cross_step",
    x_title="Char ratio (vs Claude Read)",
    y_title="sec/page",
    z_title="Cross-step value (0-12)",
    title_main="Verzió C — Char ratio × log-time × cross-step value",
)
# Detail plot per-page char_ratio + broadcast cross_step
for i, backend in enumerate(["tesseract","pymupdf4llm","mineru","claude_read"]):
    sub = df[df["backend"]==backend]
    fig_c.data[4+i].x = sub["char_ratio_to_claude_read"]
    fig_c.data[4+i].z = sub["cross_step"]
fig_c.show()

**Verzió C értelmezése:**

- A detail plotban a 4 backend Z-szinten *különböző síkokat* foglal el (cross_step backend-szintű konstans → minden 15 page-en ugyanazon a Z-szinten van), így a Z-tengely **réteg-szerű**.
- Az X-tengely viszont per-page varianciát mutat: a **Tesseract** nagyi-oldalain (magyar) az X ≈ 0.5, míg a chattopadhyay-en ≈ 1.0 — látható a magyar diakritika-veszteség.
- A **MinerU** detail-pontjaira a char_ratio nem ad direkt összevetést, mert a MinerU output forrás-szintű markdown (nem per-page text fájl). A `metrics.json`-ban a per-page chars hiányzik MinerU-ra → ezért a 4 MinerU pont az detail-plotban a default Y-en lehet csak. (Ha akarjuk per-page char-ot, parse-olni kell a `_content_list.json`-t `page_idx` szerint.)

---
## 5. Utómunka szükségessége és bizonytalanság

Mind a 3 verzió ugyanazt mondja **másképp**, de néhány *közös* bizonytalanság-forrás van. Az alábbi tábla összefoglalja:

In [ ]:
uncertainty = pd.DataFrame([
    {"forrás": "Manual scores", "backend": "mind",
     "jellegű bizonytalanság": "szakértői ítélet, n=4 backend × 7 dim, 15 oldal mintán",
     "érzékenység": "±0.5 pont; MinerU hu quality 2→3 frissítés a háttér-futás után — ez NEM elhanyagolható",
     "javaslat": "külső reviewer (😎) score-okat vizsgálja át"},
    {"forrás": "Claude Read time", "backend": "claude_read",
     "jellegű bizonytalanság": "nem mért — csak render-time a runnerben",
     "érzékenység": "a tényleges kézi-gépelés ~30-60s/oldal; a plot félrevezet, ha mérten gyorsnak látszik",
     "javaslat": "hover-tipben ⚠️; a plot magyarázó cella jelölje"},
    {"forrás": "MinerU time", "backend": "mineru",
     "jellegű bizonytalanság": "range-averaged (forrás-szintű batch / oldal)",
     "érzékenység": "egy izolált oldal NEM ~28s, mert a model-init ~11s, így 1 oldalra 50s+, 30 oldalra 30s/oldal",
     "javaslat": "batch-szerűen futtatni érdemes; rögzítjük a decision.md-ben"},
    {"forrás": "Per-page minőség", "backend": "mind",
     "jellegű bizonytalanság": "manual score backend-szintű, broadcast-olt page-szintre",
     "érzékenység": "60-pontos detail plot X tengelye Verzió A-ban valójában 4 vízszintes vonal",
     "javaslat": "page-szintű manual scoring future work"},
    {"forrás": "Sample size", "backend": "mind",
     "jellegű bizonytalanság": "15 oldal, 4 forrás",
     "érzékenység": "szignifikancia-test nincs (n túl kicsi); irány-jelzés OK",
     "javaslat": "következő hetekkel bővíteni a benchmarkot"},
    {"forrás": "Cross-step Σ", "backend": "mind",
     "jellegű bizonytalanság": "4 downstream-lépés (03/04/07/09) score-ja összegzett — egyenletes súllyal",
     "érzékenység": "súlyozás reális? Lehet, hogy 03 (mindmap) fontosabb mint 09 (tábla)",
     "javaslat": "érzékenység-elemzés súly-variálással"},
])
uncertainty

In [ ]:
# Utómunka-becslés: backend × atg/1_het 67 entry → mennyi kézi review/javítás várható
rework = pd.DataFrame([
    {"backend":"tesseract",
     "output forma":"text/pNNN.txt",
     "várható kézi rework (atg/1_het)":
        "~31 magyar oldal (nagyi) diakritika-javítása + ábra-leírás kézzel — jelentős",
     "kockázat":"low (text-only, jól lokalizálható hibák)"},
    {"backend":"pymupdf4llm",
     "output forma":"<src>.md",
     "várható kézi rework (atg/1_het)":
        "~22 scanned-oldal (gravdahl+tavakoli) teljes átírása más backenddel; egyébként minimális",
     "kockázat":"low (born-digitalon tökéletes)"},
    {"backend":"mineru",
     "output forma":"<src>.md + _content_list.json + képek",
     "várható kézi rework (atg/1_het)":
        "capture után a 02b skill futtatása (~67 entry × Read kép + 1-2 pp text-extract) — minor",
     "kockázat":"medium (kéthasábos caption-pairing helyenként téveszt; minor typo-k pl. 'Cenrtifugális')"},
    {"backend":"claude_read",
     "output forma":"text/pNNN.txt (session-szintű)",
     "várható kézi rework (atg/1_het)":
        "0 (önmagában is végleges minőség), de a session-idő 67 × ~30s ≈ 30+ perc kézi munka",
     "kockázat":"low (de skálázhatatlan automatizálás nélkül)"},
])
rework

---
## 6. Tanulság a vizualizációkból (összevetve)

1. **MinerU** és **Claude Read** mind a 3 verzió szerint Pareto-optimal `(quality=max, install=0 vs 3)`. A különbség: MinerU **batch-eable**, Claude Read **manuális**.
2. **PyMuPDF4LLM** csak born-digitalra Pareto-optimal — a scanned oldalak detail-plot-ban dramatikusan lent vannak `Z`-ben (Verzió A & C) vagy `X`-ben (Verzió B).
3. **Tesseract** középső pozícióban — *jó angol scanned-re*, *gyenge magyar text-re*. A detail-plot mutatja: a nagyi-oldalak (lila/zöld klaszter aljánál) elszakadnak a többiektől X-tengelyen.
4. **Pont méret** (impact_index = cross_step/12 + capability/5):
   - MinerU 12/12 + 5/5 = **2.0** (maximum) → legnagyobb pont
   - Claude Read 11/12 + 5/5 = **1.92** → szinte ugyanakkora
   - PyMuPDF4LLM 7/12 + 3/5 = **1.18** → közepes
   - Tesseract 3/12 + 1/5 = **0.45** → legkisebb pont
5. Az **érdekes split** mind a 3 verzióban: Y tengelyen (log-time) MinerU-Claude Read szembe; X+Z tengelyen ugyanaz. A költség-haszon választás: **install_cost** vs **manual labor** trade-off.

## 7. Mit nem mértünk meg / future work

- **Per-page MinerU char-count**: ehhez parse-olni kell a `_content_list.json` `page_idx` mezőjét. Most csak forrás-szintű chars van.
- **Per-page manual score**: a 4-pontos broadcast nem mutatja a valódi per-page minőség varianciát (pl. egyetlen rossz scan oldal Tesseractnek).
- **GPU/CPU-time bontás**: a `play_env` `mineru` env CPU-n fut. GPU-val 3-5× gyorsulás várható (`-d cuda` flag). A plot Y tengelye más lenne.
- **Marker/docTR/Surya** verseny: nincs adatpont — ha bekerül, új színek + új Pareto-front rajz.
- **Atg/1_het mind 67 entry-jén** vs. csak 15 oldal: ha minden oldalra fut a benchmark, statisztikai szignifikancia is mérhető lenne.

## 8. Hivatkozások

- [`research_design.md`](research_design.md) · [`score_template.md`](score_template.md) · [`decision.md`](decision.md)
- [`../../../test_outputs/_ocr_lab/atg_1_het/metrics.json`](../../../test_outputs/_ocr_lab/atg_1_het/metrics.json)
- [`../../../scripts/_ocr_lab_runner.py`](../../../scripts/_ocr_lab_runner.py)
- [`../../../scripts/02c_mineru_layout.py`](../../../scripts/02c_mineru_layout.py)
- [`../../../.claude/skills/02b_figure_enricher.md`](../../../.claude/skills/02b_figure_enricher.md) (v1.1)